# Lab 11 · Delegate an architecture review with A2A

**Prepared by Mustafa Assaf** · Self-study: Connect and organize your council · 15–20 minute lab

## Ask another agent to review the proposal

**So far:** one program did all the work, calling a model and its own tools.

**New:** *delegation*. Our model proposes a decision, and a separate reviewer agent checks it over the **Agent2Agent (A2A)** protocol. When the reviewer rejects a proposal, our model revises it using the reviewer's feedback.

The course helper `review_over_a2a` performs three real A2A steps with the official SDK, all inside this notebook:

1. **Start** a reviewer agent that publishes an *Agent Card* with its name, description, and skills.
2. **Discover** the reviewer by fetching that card, just as a client does for an agent it has not used before.
3. **Send** the case and the proposal as an A2A message, and read the reply: `PASS` or `FAIL` with a reason.

The reviewer applies the course policy with plain code. You can read the helper in [`coursekit/protocol_examples.py`](https://github.com/mkassaf/agentic-ai-course/blob/main/coursekit/protocol_examples.py).

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'a2a']):
    %pip install -q "agentic-ai-coursekit[a2a] @ https://github.com/mkassaf/agentic-ai-course/archive/dbe84948102e503e581fd5236bb7e67e63dba56c.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Step 1 · Call the reviewer once

We send a proposal that breaks the policy, to see a rejection. `report.partition(": ")` splits the report at the first `": "` into the status and the feedback.

In [ ]:
from coursekit.protocol_examples import review_over_a2a

case = {"team_size": 9, "independent_deploys": False}
agent, report = await review_over_a2a(case, "microservices")
status, _, feedback = report.partition(": ")
print("Discovered agent:", agent)
print("Status:", status)
print("Feedback:", feedback)

## Step 2 · A proposer that can revise

`propose` asks our model for a decision. On a retry, it adds the reviewer's feedback to the prompt. The prompt does not contain the policy, so the model can make the same mistakes as in Lab 00. Here the reviewer catches them.

In practice mode, `scripted` holds the replies for each project: the first attempt, then the revision if there is one.

In [ ]:
projects = [
    {"name": "Course portal",     "team_size": 3,  "independent_deploys": False},
    {"name": "Payments platform", "team_size": 12, "independent_deploys": True},
    {"name": "Internal wiki",     "team_size": 9,  "independent_deploys": False},
    {"name": "Mobile backend",    "team_size": 5,  "independent_deploys": True},
]
scripted = {"Course portal": ["modular monolith"], "Payments platform": ["microservices"],
            "Internal wiki": ["microservices", "modular monolith"], "Mobile backend": ["modular monolith"]}

def propose(project, feedback=""):
    """Ask the model for a decision. On a retry, include the reviewer's feedback."""
    prompt = f"Pick modular monolith or microservices for {project}. Reply with the label only."
    if feedback:
        prompt += f"\nA reviewer rejected your last answer: {feedback} Answer again."
    attempt = 1 if feedback else 0
    reply = ask(prompt, example=scripted[project["name"]][attempt])
    return reply.strip().rstrip(".").lower()  # "Modular monolith." -> "modular monolith"

## Step 3 · Propose, review, and revise

For each project, our model proposes and the reviewer checks. A rejected proposal gets exactly one revision, so the exchange always ends. `passed` records the final verdict for each project, and `revisions` counts the corrections.

In [ ]:
passed, revisions = {}, 0
for project in projects:
    proposal = propose(project)
    _, report = await review_over_a2a(project, proposal)
    print(f"{project['name']:<18} proposed {proposal!r:<19} -> {report}")
    if report.startswith("FAIL"):
        revisions += 1
        proposal = propose(project, feedback=report)
        _, report = await review_over_a2a(project, proposal)
        print(f"{'':<18} revised  {proposal!r:<19} -> {report}")
    passed[project["name"]] = report.startswith("PASS")

## Step 4 · Check the outcome

Every project should pass after at most one revision. The revision count shows how often the reviewer had to correct our model.

In [ ]:
print(passed)
print("Revisions needed:", revisions)
assert all(passed.values()), "A proposal still fails after one revision."

## What you should see

Step 1 prints `Discovered agent: Architecture reviewer`, `Status: FAIL`, and `Feedback: course policy expects modular monolith.` In Step 3, three projects pass at once. The Internal wiki is first proposed as `'microservices'` and fails, then the revision `'modular monolith'` passes. Step 4 shows every project as `True` and `Revisions needed: 1`.

## Try it

1. Make the Mobile backend's first scripted reply `"microservices"`, and add `"modular monolith"` as its revision. Run Steps 2 to 4. Now two projects need a revision.
2. Make a revision wrong too, for example `["microservices", "microservices"]`. The final check stops the run.
3. In live mode, the model's revisions are real. Does a real model take the reviewer's feedback?
4. **Use it in your council.** Run one architect from Lab 05 as an A2A agent with its own Agent Card. What must each message carry so that the chair's transcript stays traceable?

## Takeaway

A2A lets one agent find another agent through its Agent Card and exchange messages with it, without sharing code. A separate reviewer gives a second, independent check, and its feedback can drive a revision. Lab 12 builds the same choose, check, and retry loop with a framework.

## Read further

- [Official A2A SDK](https://github.com/a2aproject/a2a-python)
- [Agent discovery](https://a2a-protocol.org/latest/topics/agent-discovery/)